# Resumable Colab development session

This notebook prepares one bounded stochastic development model using the checked-in **learning pilot v3** protocol. It preserves independent User Vehicle decisions, the physical models, and standard learned-epsilon behavior. The default model is QR-DQN + CVaR with the individual return scope. It is not a paper experiment or main training; the main protocol still needs to be frozen and approved. There is no main-training or held-out-test launch cell.

All execution gates default to `False`. **Run all does not mount Drive or start training.** Source verification and metadata preflight run only after you supply the source identity. Start with a CPU runtime; a selected Colab GPU does not accelerate this CPU driver.


## Prepare the exact private source

After this milestone is committed and pushed, obtain the full commit and Python-source digest from that same trusted checkout:

```text
git rev-parse HEAD
python -c "from experiments.provenance import source_provenance; print(source_provenance()['source_sha256'])"
```

While signed in to GitHub **in your browser**, download `https://github.com/Amirr-1378/risk-aware-vehicular-edge-computing/archive/<FULL_COMMIT>.zip`. Upload that ZIP through Colab's Files sidebar and set `ARCHIVE_PATH` to its uploaded path, or rename it to `source.zip`. Copy the independently obtained commit and digest into the configuration below. GitHub credentials remain in the browser; do not put a password or token in notebook cells, URLs, environment variables, or outputs.

The archive root is checked against the expected commit; the root name alone does not prove Git history. The independently trusted digest checks Python files under `src/` and directly under `experiments/`, with normalized text newlines. Configurations, documentation and `pyproject.toml` are outside that digest: obtain the entire archive from the trusted browser download. The runner separately binds the exact scientific protocols. Start a fresh runtime when changing source versions, and use a new source destination; existing extracted source is never overwritten.


In [ ]:
# Fill source identity before running the setup cells. No credentials belong here.
EXPECTED_COMMIT = ""
EXPECTED_SOURCE_SHA256 = ""
ARCHIVE_PATH = "/content/source.zip"
SOURCE_DESTINATION = "/content/risk-aware-vehicular-edge-computing-source"

# Choose a fresh Drive subfolder for a new session; keep it for explicit resume.
OUTPUT_DIRECTORY = "/content/drive/MyDrive/risk-aware-v2x/development-session-001"
ALGORITHM = "qr_dqn_cvar"
AGENT_SEED = 11
RETURN_SCOPE = "individual"
MAX_ADDITIONAL_EPISODES = 1
RESUME = False

# Opt in separately after inspecting setup and preflight results.
RUN_TESTS = False
MOUNT_DRIVE = False
RUN_DEVELOPMENT = False
PREFLIGHT_READY = False
DRIVE_READY = False


## Verify and extract source locally

The following bootstrap embeds the repository's standalone, standard-library ZIP verifier verbatim, since the source tree does not exist yet. It validates every archive member and the source digest before extraction. It makes no network request and installs no packages. Empty identity placeholders deliberately stop setup.


In [ ]:
from pathlib import Path
import os
import sys

SOURCE_HELPER_CODE = "\"\"\"Validate and extract a browser-downloaded private GitHub source ZIP offline.\n\nThe archive root declares the expected commit; it does not cryptographically\nprove Git history. A separately trusted SHA256 verifies the Python source set\nusing the same text normalization as ``source_provenance``. No credentials,\nnetwork requests, dependency installation or training are involved.\n\"\"\"\n\nimport argparse\nimport hashlib\nimport json\nimport re\nimport stat\nfrom pathlib import Path, PurePosixPath\nfrom zipfile import ZipFile\n\n\n_MAX_EXTRACTED_BYTES = 100 * 1024 * 1024\n_REQUIRED_FILES = frozenset({\"README.md\", \"AGENTS.md\", \"pyproject.toml\",\n    \"src/risk_aware_v2x/contracts.py\", \"experiments/provenance.py\"})\n_WINDOWS_RESERVED = frozenset({\"CON\", \"PRN\", \"AUX\", \"NUL\", \"CONIN$\", \"CONOUT$\",\n    *(f\"COM{i}\" for i in range(1, 10)), *(f\"LPT{i}\" for i in range(1, 10))})\n\n\ndef _member_path(info, expected_root):\n    name = info.filename\n    if \"\\x00\" in info.orig_filename or \"\\\\\" in info.orig_filename or name.startswith(\"/\"):\n        raise ValueError(\"archive contains an unsafe path\")\n    pieces = name.rstrip(\"/\").split(\"/\")\n    if any(not piece or piece in (\".\", \"..\") or any(char in '<>:\"|?*' or ord(char) < 32 for char in piece)\n           or piece.rstrip(\" .\") != piece\n           or piece.split(\".\")[0].upper() in _WINDOWS_RESERVED for piece in pieces):\n        raise ValueError(\"archive contains an unsafe or nonportable path\")\n    if pieces[0] != expected_root:\n        raise ValueError(\"archive must have exactly the expected commit root\")\n    if any(piece.casefold() == \".git\" for piece in pieces[1:]):\n        raise ValueError(\"archive must not contain a .git tree\")\n    mode_type = stat.S_IFMT((info.external_attr >> 16) & 0xFFFF)\n    allowed_type = stat.S_IFDIR if info.is_dir() else stat.S_IFREG\n    if mode_type not in (0, allowed_type):\n        raise ValueError(\"archive members must be regular files or directories\")\n    if info.flag_bits & 1:\n        raise ValueError(\"encrypted archive members are unsupported\")\n    if len(pieces) == 1 and not info.is_dir():\n        raise ValueError(\"archive root must be a directory\")\n    if info.is_dir() and info.file_size:\n        raise ValueError(\"archive directory must not contain file data\")\n    return PurePosixPath(*pieces[1:]).as_posix() if len(pieces) > 1 else \"\"\n\n\ndef _source_sha256(files):\n    digest = hashlib.sha256()\n    for relative in sorted(files):\n        parts = PurePosixPath(relative).parts\n        is_source = relative.endswith(\".py\") and (\n            parts[0] == \"src\" or (parts[0] == \"experiments\" and len(parts) == 2))\n        if is_source:\n            # Path.read_text(encoding='utf-8') uses universal newline handling.\n            text = files[relative].decode(\"utf-8\").replace(\"\\r\\n\", \"\\n\").replace(\"\\r\", \"\\n\")\n            digest.update(relative.encode(\"utf-8\") + b\"\\0\")\n            digest.update(text.encode(\"utf-8\") + b\"\\0\")\n    return digest.hexdigest()\n\n\ndef extract_source_archive(archive: Path, destination: Path, *, expected_commit: str,\n                           expected_source_sha256: str) -> Path:\n    \"\"\"Validate every member and source digest before creating a new root.\n\n    The returned root is ``destination`` with the single GitHub archive prefix\n    stripped. Existing paths are never replaced. The 100 MiB limit applies to\n    the sum of uncompressed member sizes. Non-Python contents are presence/\n    path checked, but are outside the source-provenance SHA256 contract.\n    \"\"\"\n    if not isinstance(expected_commit, str) or not re.fullmatch(r\"[0-9a-fA-F]{40}\", expected_commit):\n        raise ValueError(\"expected_commit must be a full 40-hex commit\")\n    if not isinstance(expected_source_sha256, str) or not re.fullmatch(r\"[0-9a-fA-F]{64}\", expected_source_sha256):\n        raise ValueError(\"expected_source_sha256 must be a trusted 64-hex digest\")\n    destination = Path(destination)\n    if destination.exists() or destination.is_symlink():\n        raise FileExistsError(\"choose a new source destination\")\n    expected_root = \"risk-aware-vehicular-edge-computing-\" + expected_commit\n    files, directories, seen, spellings = {}, set(), set(), {}\n    with ZipFile(archive) as source:\n        members = source.infolist()\n        if sum(member.file_size for member in members) > _MAX_EXTRACTED_BYTES:\n            raise ValueError(\"archive exceeds the 100 MiB extracted-size limit\")\n        for member in members:\n            relative = _member_path(member, expected_root)\n            portable_key = relative.casefold()\n            if portable_key in seen:\n                raise ValueError(\"archive contains duplicate or case-colliding members\")\n            seen.add(portable_key)\n            parts = PurePosixPath(relative).parts\n            for count in range(1, len(parts) + 1):\n                prefix = \"/\".join(parts[:count])\n                key = prefix.casefold()\n                if key in spellings and spellings[key] != prefix:\n                    raise ValueError(\"archive contains case-colliding directory paths\")\n                spellings[key] = prefix\n            if member.is_dir():\n                if relative:\n                    directories.add(relative)\n            else:\n                # Reading all bytes also checks CRC/decompression before any writes.\n                files[relative] = source.read(member)\n        file_keys = {relative.casefold() for relative in files}\n        for relative in (*files, *directories):\n            for parent in PurePosixPath(relative).parents:\n                if parent.as_posix().casefold() in file_keys:\n                    raise ValueError(\"archive file conflicts with a directory\")\n        if not _REQUIRED_FILES.issubset(files):\n            raise ValueError(\"archive is missing required repository files\")\n        if _source_sha256(files) != expected_source_sha256.lower():\n            raise ValueError(\"archive Python sources do not match the trusted source digest\")\n\n    destination.mkdir(parents=True, exist_ok=False)\n    for relative in sorted(directories, key=lambda name: (name.count(\"/\"), name)):\n        destination.joinpath(*PurePosixPath(relative).parts).mkdir(parents=True, exist_ok=True)\n    for relative, data in files.items():\n        target = destination.joinpath(*PurePosixPath(relative).parts)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        with target.open(\"xb\") as output:\n            output.write(data)\n    return destination\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--archive\", type=Path, required=True)\n    parser.add_argument(\"--destination\", type=Path, required=True)\n    parser.add_argument(\"--commit\", required=True)\n    parser.add_argument(\"--source-sha256\", required=True)\n    args = parser.parse_args(argv)\n    root = extract_source_archive(args.archive, args.destination,\n        expected_commit=args.commit, expected_source_sha256=args.source_sha256)\n    print(json.dumps({\"root\": str(root)}))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
bootstrap_namespace = {"__name__": "colab_source_bootstrap"}
exec(compile(SOURCE_HELPER_CODE, "colab_source_bootstrap.py", "exec"), bootstrap_namespace)
REPO = bootstrap_namespace["extract_source_archive"](
    Path(ARCHIVE_PATH), Path(SOURCE_DESTINATION),
    expected_commit=EXPECTED_COMMIT,
    expected_source_sha256=EXPECTED_SOURCE_SHA256,
)
os.chdir(REPO)
for source_path in (str(REPO / "src"), str(REPO)):
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
os.environ["PYTHONPATH"] = os.pathsep.join((str(REPO), str(REPO / "src")))
print(f"Verified source extracted to {REPO}")


## Inspect the runtime before opting in

Preflight only checks installed metadata; it does not import Torch/NumPy, install dependencies, train, evaluate, or inspect test trajectories. It requires Python >=3.11, stable Torch >=2.6,<3, and stable NumPy >=1.26,<3. If checks fail, stop and explicitly choose or prepare a compatible runtime using `pyproject.toml`; there is no automatic installation cell. A later learning import can still expose an environment problem not detectable from package metadata.

Resume additionally requires the original exact source, protocol, encoder, algorithm/seed/scope and runtime identity. Broadly compatible version ranges are insufficient for resuming an existing session. A replacement Colab VM may require explicitly restoring the original runtime rather than continuing with different versions.


In [ ]:
import json
from experiments.runtime_preflight import collect_preflight

PREFLIGHT_READY = False
PREFLIGHT = collect_preflight(require_learning=True)
print(json.dumps(PREFLIGHT, indent=2, allow_nan=False))
if PREFLIGHT["provenance"]["source_sha256"] != EXPECTED_SOURCE_SHA256.lower():
    raise RuntimeError("Extracted/imported Python source does not match the expected digest.")
if not PREFLIGHT["ok"]:
    raise RuntimeError("Runtime preflight failed; prepare compatible dependencies explicitly before continuing.")
PREFLIGHT_READY = True


## Optional offline setup checks

Set `RUN_TESTS=True` to run only the notebook, source-archive and metadata-preflight tests using standard-library unittest discovery. These checks do not train an agent or mount Drive and require no additional test package. The full repository test suite also contains small learning smoke tests, so it is deliberately not launched by this notebook.


In [ ]:
if RUN_TESTS:
    import subprocess
    import sys
    for pattern in ("test_colab*.py", "test_runtime_preflight.py"):
        subprocess.run([
            sys.executable, "-m", "unittest", "discover",
            "-s", "tests", "-p", pattern, "-v",
        ], cwd=REPO, check=True)
else:
    print("Offline setup checks skipped: RUN_TESTS=False.")


## Explicitly mount persistent output storage

Only set `MOUNT_DRIVE=True` when ready to authorize Colab's Drive prompt. Use an absolute output path inside your mounted `MyDrive`, with a new subfolder for a new session. Do not run two tabs or runtimes against the same session folder: the journal assumes one writer.

Drive synchronization is best effort. Immutable step files, a commit record written last and hash validation detect incomplete or corrupt writes; they do not establish cloud-level atomic durability after a disconnect. Preserve original session folders and wait for Drive synchronization before ending a runtime.


In [ ]:
DRIVE_READY = False
if MOUNT_DRIVE:
    from pathlib import Path
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_READY = Path("/content/drive/MyDrive").is_dir()
    if not DRIVE_READY:
        raise RuntimeError("Drive MyDrive is unavailable after mounting.")
else:
    print("Drive not mounted: MOUNT_DRIVE=False.")


## Explicitly advance one development model

After inspecting preflight and storage, set `RUN_DEVELOPMENT=True` and run the cell below. It starts a fresh session only with `RESUME=False`; an existing session requires `RESUME=True`. Each invocation permits at most **one additional complete training episode** by default. Later invocations resume the same fixed pilot budget; increasing that fixed budget or changing scenes/runtime is not a valid resume.

The journal starts from an untrained checkpoint and commits each full episode with its report and checkpoint. Once the fixed final episode is reached, the runner performs only the protocol's development validation with a frozen model and publishes the final validation report. It never selects a best checkpoint from validation, uses held-out test scenes, or launches main training. Standard learned-epsilon behavior is used; the optional greedy warmup is not enabled.

The cell prints a compact progress summary. Full results remain in `DEVELOPMENT_RESULT`, and complete reports are retained in the journal. A partial session is incomplete development evidence. Keep its completed/planned counts visible and do not compare it as a finished paper result against greedy or a full training run.


In [ ]:
if RUN_DEVELOPMENT:
    if not PREFLIGHT_READY:
        raise RuntimeError("Run the verified-source setup and successful runtime preflight first.")
    if not MOUNT_DRIVE or not DRIVE_READY:
        raise RuntimeError("Explicitly mount Drive and confirm MyDrive before development execution.")
    from pathlib import Path
    import json
    output_path = Path(OUTPUT_DIRECTORY)
    drive_root = Path("/content/drive/MyDrive")
    if (not output_path.is_absolute() or output_path == drive_root
            or not output_path.resolve().is_relative_to(drive_root.resolve())):
        raise ValueError("Choose an absolute development-session subfolder inside Drive MyDrive.")
    from experiments.run_resumable_development import run_resumable_development
    DEVELOPMENT_RESULT = run_resumable_development(
        output_directory=output_path,
        algorithm=ALGORITHM,
        agent_seed=AGENT_SEED,
        return_scope=RETURN_SCOPE,
        resume=RESUME,
        max_additional_episodes=MAX_ADDITIONAL_EPISODES,
    )
    summary_keys = (
        "status", "completed_training_episodes", "planned_training_episodes",
        "completed_validation_episodes", "planned_validation_episodes",
        "training_slots", "seen_transitions", "gradient_updates", "checkpoint",
        "model_sha256", "main_training_started", "test_split_accessed",
    )
    DEVELOPMENT_SUMMARY = {key: DEVELOPMENT_RESULT[key] for key in summary_keys}
    print(json.dumps(DEVELOPMENT_SUMMARY, indent=2, allow_nan=False, default=str))
else:
    print("Development execution skipped: RUN_DEVELOPMENT=False.")


## Continue or recover explicitly

For normal continuation, restore the same verified source and exact runtime, mount the same Drive, set `RESUME=True`, and run the development cell again. Resume loads only a complete, contiguous committed episode boundary. Source setup is exclusive: within an unchanged runtime, reuse the already verified source and rerun preflight/Drive/development cells rather than extracting over it.

If the runner detects an incomplete final step or an uncommitted file, it stops without deleting or overwriting the source session. Do not remove journal files by hand. The explicit `experiments.session_journal.copy_verified_prefix(source, fresh_destination, exact_original_run_contract)` recovery utility can copy a verified committed prefix to a **new** output folder while retaining the original. Only an incomplete final generation may be excluded; a corrupt committed step is rejected. Recovery is intentionally not executed by this notebook. Resume the recovered folder only after verifying its report and unchanged contract.

Keep the session's complete artifacts, protocol identity and runtime report. No main training has been started by preparing this notebook. Main training needs a separately frozen protocol and an explicit start decision in Colab.
